## Practice 3 Hugging Face Transformers: Sentiment Analysis and Binary Text Classification

**General Objective:**  
Become familiar with the Hugging Face Transformers ecosystem through two exercises:
- **Exercise 1:** Use an already fine-tuned model to perform inference.
- **Exercise 2:** Fine-tune a generic pretrained model for binary text classification.

### 1. Objective

- Distinguish between the two layers of knowledge: **Pretraining** (language knowledge) và **Downstream Fine-tuning**
  (sentiment classification knowledge).

| Exercise | Main Objective | Training Required? | Checkpoint Used |
|----------|----------------|--------------------|--------------------|
| **Exercise 1** | Inference + tokenizer understanding | Không | `distilbert-base-uncased-finetuned-sst-2-english` |
| **Exercise 2** | Full-process fine-tuning | Yes | `distilbert-base-uncased` (generic) |


### 2. Scientific and Theoretical Foundation

#### (a) Why Choose Rotten Tomatoes

| Criterion | Rotten Tomatoes | IMDb |
|---|---|---|
| Labeled samples | 10,662 | 50,000 |
| Train split | 8,530 | 25,000 |
| Validation split | 1,066 | No separate official split |
| Test split | 1,066 | 25,000 |
| Binary sentiment | Yes | Yes |
| Typical text length | Short | Longer |
| Fine-tuning cost | Lower | High hơn |
| Suitable for a compact practice notebook | Very high | High |
| Alignment with the official Hugging Face tutorial | Medium | Very high |

**Reason for choosing:** a binary classification task, a moderate computational workload for a practice notebook, three existing
train/validation/test splits, relatively short reviews, no need to create an additional validation split,
and support for a clean experimental workflow on a personal computer (CPU-only).

```
Train: 8,530
Validation: 1,066
Test: 1,066
Total: 10,662

Label 0: NEGATIVE
Label 1: POSITIVE
```

#### (b) Why Choose DistilBERT

```
flowchart LR
    A[Tokenized Text] --> B[DistilBERT Backbone]
    B --> C[Contextual Representation]
    C --> D[Classification Head]
    D --> E[Two Logits]
    E --> F[NEGATIVE or POSITIVE]
```

| Model | Main Advantages | Main Limitations |
|---|---|---|
| DistilBERT | Lightweight, practical | Lower capacity than BERT-base |
| BERT-base | Classic Transformer baseline | Higher computational cost |
| RoBERTa-base | Strong language representation | Higher computational cost |
| MiniLM | Very lightweight | Less aligned with the traditional introductory workflow |
| ALBERT | Parameter-efficient | Different architectural characteristics |

**Primary implementation choice: DistilBERT** - a pretrained Transformer that is lighter than BERT-base, directly supports
sequence classification, is suitable for English sentiment analysis, reduces training cost while preserving
the nature of the transfer-learning workflow, and is suitable for a CPU-only lab environment.

#### (c) Conceptual Nature of Transfer Learning

```
flowchart TD
    A[Pretraining] --> B[General Language Knowledge]
    B --> C[Downstream Fine-Tuning]
    C --> D[Binary Sentiment Knowledge]
    D --> E[Inference]
    E --> F[Positive or Negative Prediction]
```

```
Exercise 1
Already fine-tuned model
        |
        v
Inference

Exercise 2
Generic pretrained model
        |
        v
Task-specific fine-tuning
        |
        v
Binary classifier
```

#### (d) How Fine-tuning Differs from Feature Extraction

```
Feature extraction
Freeze Transformer backbone
        |
        v
Train only classifier

Fine-tuning
Update Transformer backbone
        +
Update classification head
```

| Criterion | Feature Extraction | Fine-tuning |
|----------|--------------------|-----------------------------------|
| Base model weights | Freeze (not updated) | Updated together with the classification head |
| Base model role | Only extracts fixed features | Adapts representations to fit the task |
| Computational cost | Lower | High hơn |
| Application in this practice | Not used | Yes (Trainer fine-tunes the entire model) |

Practice 3 (Exercise 2) applies **Fine-tuning** to the entire `distilbert-base-uncased` backbone, which is updated
together with the classification head; no layers are frozen.

#### (e) Things That Must Never Be Done Throughout Practice 3

```
Do not:
Fine-tune on the test set

Do not:
Use test performance to choose epochs or hyperparameters

Do not:
Aggressively remove stopwords, punctuation, or linguistic structure without justification

Do not:
Train DistilBERT from scratch for this practice

Do not:
Use an already sentiment-fine-tuned checkpoint in Exercise 2
and describe the process as generic pretrained-model fine-tuning
without explicitly documenting the checkpoint's prior task-specific training

Do not:
Fabricate loss curves, metrics, confusion matrices, or benchmark results
```

### 3. Overall End-to-End Pipeline Map

```
flowchart TD
    S[START] --> A[Environment and Seeds]

    A --> B[Exercise 1]
    B --> C[Load Fine-Tuned Sentiment Model]
    C --> D[Inspect Sentence Tokens]
    D --> E[Run Sentiment Inference]

    E --> F[Exercise 2]
    F --> G[Load Rotten Tomatoes Dataset]
    G --> H[EDA and Data Quality Checks]
    H --> I[Load DistilBERT Tokenizer]
    I --> J[Token-Length Analysis]
    J --> K[Tokenize Dataset]
    K --> L[Dynamic Padding]
    L --> M[Load Pretrained DistilBERT Classifier]
    M --> N[Model Sanity Check]
    N --> O[Define Metrics]
    O --> P[Configure TrainingArguments]
    P --> Q[Create Trainer]
    Q --> R[Debug Subset Run]
    R --> T{Pipeline Valid?}
    T -- No --> U[Fix and Re-run]
    U --> R
    T -- Yes --> V[Full Fine-Tuning]
    V --> W[Validation Monitoring]
    W --> X[Load Best Checkpoint]
    X --> Y[Final Test Evaluation]
    Y --> Z[Confusion Matrix]
    Z --> AA[Error Analysis]
    AA --> AB[New-Sentence Inference]
    AB --> AC[Save Model and Tokenizer]
    AC --> AD[Reload Sanity Test]
    AD --> AE[FINAL SUMMARY]
```

### 4. 16-Phase Diagram According to Notebook Architecture

```
flowchart TD
    P0[Phase 0<br/>Practice Overview] --> P1[Phase 1<br/>Environment and Reproducibility]
    P1 --> P2[Phase 2<br/>Pretrained Sentiment Inference]
    P2 --> P3[Phase 3<br/>Tokenization Investigation]
    P3 --> P4[Phase 4<br/>Dataset Loading]
    P4 --> P5[Phase 5<br/>EDA and Sanity Checks]
    P5 --> P6[Phase 6<br/>Tokenizer and Preprocessing]
    P6 --> P7[Phase 7<br/>Model Construction]
    P7 --> P8[Phase 8<br/>Metrics and Training Configuration]
    P8 --> P9[Phase 9<br/>Fine-Tuning]
    P9 --> P10[Phase 10<br/>Learning Curves]
    P10 --> P11[Phase 11<br/>Validation and Test Evaluation]
    P11 --> P12[Phase 12<br/>Error Analysis]
    P12 --> P13[Phase 13<br/>New-Sentence Inference]
    P13 --> P14[Phase 14<br/>Save and Reload]
    P14 --> P15[Phase 15<br/>Final Summary]
```

### 5. Stage 1 Scope

| Phase | Phase Name | Main Objective |
|-------|-----------|----------------|
| 0 | Practice Overview | Problem definition + academic architecture |
| 1 | Environment & Reproducibility | Set up environment, seed, device |
| 2 | Pretrained Sentiment Inference | Exercise 1 - Inference |
| 3 | Tokenization Investigation | Tokenizer analysis |
| 4 | Dataset Loading | Load Rotten Tomatoes |
| 5 | Dataset EDA & Sanity Checks | Data analysis |
| 6 | Tokenizer & Preprocessing | Prepare data for fine-tuning |


### 6. Key Technical Decisions

| Item | Decision | Status |
|----------|----------|--------|
| Model Exercise 1 | `distilbert-base-uncased-finetuned-sst-2-english` | Approved |
| Model Exercise 2 | `distilbert-base-uncased` | Approved |
| Dataset | Rotten Tomatoes | Approved |
| Hardware | CPU only | Approved |
| Random Seed | 42 | Approved |

### 7. Final Conceptual Summary

```
flowchart LR
    A[General Language Pretraining] --> B[Pretrained DistilBERT]
    B --> C[Binary Sentiment Fine-Tuning]
    C --> D[Task-Specific Classifier]
    D --> E[Validation]
    E --> F[Independent Test Evaluation]
    F --> G[Reusable Sentiment Model]
```

```
Exercise 1
Pretrained model reuse
        +
Tokenizer understanding

Exercise 2
Transfer learning
        +
Fine-tuning
        +
Scientific evaluation
        +
Reusable model artifacts
```